# COVID Largo: Síntomas Neuropsiquiátricos — Datos Globales (2021–2024)

**DOI:** [![DOI](https://img.shields.io/badge/DOI-10.7910%2FDVN%2FQUWC1F-blue)](https://doi.org/10.7910/DVN/QUWC1F)  
**Autor:** Juan Moises de la Serna | ORCID: [0000-0002-8401-8018](https://orcid.org/0000-0002-8401-8018)  
**Licencia:** CC0 1.0

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanmoisesd/covid-largo-sintomas-neuropsiquiatricos-prevalencia-2021-2024/blob/main/notebooks/covid_largo_neuropsiquiatrico_analisis.ipynb)

## Descripción
Análisis de manifestaciones neuropsiquiátricas en pacientes con COVID Largo: niebla mental, depresión, ansiedad y deterioro cognitivo.

**Estadísticas clave:**
- **Niebla mental**: 22–32% de los pacientes con COVID Largo
- **Depresión**: prevalencia del 23–26% en COVID Largo
- **Ansiedad**: prevalencia del 22–24%
- **Deterioro cognitivo**: 20–30% (hasta 3× lo normal)
- Los síntomas persisten 3–12+ meses tras la infección

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
print('✓ Bibliotecas cargadas correctamente')

In [ ]:
# ─── Conjunto de datos simulado ────────────────────────────────────────────────
np.random.seed(2024)
n = 500
estado = np.random.choice(['COVID_Largo','Recuperado','Control'], n, p=[0.35,0.35,0.30])

df = pd.DataFrame({
    'id_sujeto': [f'CL{i:04d}' for i in range(1, n+1)],
    'estado': estado,
    'edad': np.random.randint(18, 72, n),
    'sexo': np.random.choice(['F','M'], n, p=[0.58,0.42]),
    'pais': np.random.choice(['España','México','Argentina','Colombia','Brasil','Italia'], n),
    'meses_post_covid': np.where(estado=='Control', 0, np.random.choice(range(1,25), n)),
    'niebla_mental': np.where(estado=='COVID_Largo', np.random.binomial(1,0.27,n),
                      np.where(estado=='Recuperado', np.random.binomial(1,0.09,n), np.random.binomial(1,0.05,n))),
    'puntuacion_PHQ9': np.where(estado=='COVID_Largo', np.random.normal(10.8,4.2,n),
                        np.where(estado=='Recuperado', np.random.normal(5.1,3.3,n), np.random.normal(3.2,2.8,n))).clip(min=0,max=27),
    'puntuacion_GAD7': np.where(estado=='COVID_Largo', np.random.normal(9.3,4.0,n),
                        np.where(estado=='Recuperado', np.random.normal(4.5,3.1,n), np.random.normal(2.9,2.5,n))).clip(min=0,max=21),
    'puntuacion_cognitiva': np.where(estado=='COVID_Largo', np.random.normal(62,12,n),
                             np.where(estado=='Recuperado', np.random.normal(78,9,n), np.random.normal(85,8,n))).clip(min=0,max=100),
    'puntuacion_fatiga': np.where(estado=='COVID_Largo', np.random.normal(7.1,2.0,n),
                          np.where(estado=='Recuperado', np.random.normal(3.9,2.1,n), np.random.normal(2.1,1.6,n))).clip(min=0,max=10)
})

print(f'Dimensiones: {df.shape}')
print(f'Distribución por estado:\n{df.estado.value_counts()}')
df.head()

In [ ]:
# ─── Figura 1: Comparación de síntomas por grupo ─────────────────────────────
fig, ejes = plt.subplots(2, 2, figsize=(14, 10))
metricas = ['puntuacion_PHQ9', 'puntuacion_GAD7', 'puntuacion_cognitiva', 'puntuacion_fatiga']
titulos  = ['Depresión (PHQ-9)', 'Ansiedad (GAD-7)', 'Puntuación Cognitiva', 'Puntuación de Fatiga']
paleta = {'COVID_Largo': '#E74C3C', 'Recuperado': '#F39C12', 'Control': '#27AE60'}
orden = ['Control', 'Recuperado', 'COVID_Largo']

for eje, metrica, titulo in zip(ejes.flat, metricas, titulos):
    sns.violinplot(data=df, x='estado', y=metrica, order=orden, palette=paleta, ax=eje, cut=0, linewidth=1)
    sns.stripplot(data=df, x='estado', y=metrica, order=orden, palette=paleta, ax=eje, size=1.5, alpha=0.25)
    eje.set_title(titulo, fontweight='bold', fontsize=12)
    eje.set_xlabel('')
    eje.set_xticklabels(['Control','Recuperado','COVID Largo'], fontsize=9)

plt.suptitle('Síntomas Neuropsiquiátricos: COVID Largo vs Grupos Control\nDOI: 10.7910/DVN/QUWC1F',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_comparacion_sintomas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Figura 2: Evolución de síntomas con el tiempo ────────────────────────────
df_cl = df[df['estado']=='COVID_Largo'].copy()
df_cl['intervalo_mes'] = pd.cut(df_cl['meses_post_covid'],
    bins=[0,3,6,9,12,24], labels=['1-3m','4-6m','7-9m','10-12m','>12m'])

fig, eje = plt.subplots(figsize=(11, 6))
for metrica, color, marcador in [
    ('puntuacion_PHQ9','#E74C3C','o'), ('puntuacion_GAD7','#F39C12','s'),
    ('puntuacion_fatiga','#9B59B6','^')]:
    gb = df_cl.groupby('intervalo_mes', observed=True)[metrica].agg(['mean','sem'])
    eje.errorbar(range(len(gb)), gb['mean'], yerr=gb['sem'], fmt=f'-{marcador}',
                 color=color, lw=2, markersize=8, capsize=4, label=metrica.replace('_',' ').title())

eje.set_xticks(range(5))
eje.set_xticklabels(['1-3m','4-6m','7-9m','10-12m','>12m'], fontsize=11)
eje.set_xlabel('Meses tras la infección por COVID-19', fontsize=12)
eje.set_ylabel('Puntuación media', fontsize=12)
eje.set_title('Evolución Temporal de Síntomas Neuropsiquiátricos en COVID Largo\nDOI: 10.7910/DVN/QUWC1F', fontsize=13, fontweight='bold')
eje.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig2_evolucion_sintomas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Figura 3: Prevalencia por país ───────────────────────────────────────────
datos_prev = {
    'Pais': ['EE.UU.','Reino Unido','España','Alemania','Brasil','Italia','China','Francia','India','Australia'],
    'niebla_mental_pct': [28.3,31.1,24.5,22.8,26.9,29.4,19.2,23.6,21.4,25.7],
    'depresion_pct': [25.1,26.8,22.4,20.9,28.3,24.7,17.5,21.8,23.1,22.3],
    'ansiedad_pct': [23.4,24.1,21.8,19.6,26.5,23.2,16.8,20.4,22.0,21.7]
}
df_prev = pd.DataFrame(datos_prev)

x = np.arange(len(df_prev))
ancho = 0.28
fig, eje = plt.subplots(figsize=(14,6))
eje.bar(x - ancho, df_prev['niebla_mental_pct'], ancho, label='Niebla Mental', color='#E74C3C', alpha=0.85)
eje.bar(x, df_prev['depresion_pct'], ancho, label='Depresión', color='#3498DB', alpha=0.85)
eje.bar(x + ancho, df_prev['ansiedad_pct'], ancho, label='Ansiedad', color='#27AE60', alpha=0.85)
eje.set_xticks(x)
eje.set_xticklabels(df_prev['Pais'], rotation=30, ha='right', fontsize=10)
eje.set_ylabel('Prevalencia (%)', fontsize=12)
eje.set_title('Prevalencia de Síntomas Neuropsiquiátricos en COVID Largo por País\nDOI: 10.7910/DVN/QUWC1F', fontsize=13, fontweight='bold')
eje.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig3_prevalencia_por_pais.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Comparación estadística ──────────────────────────────────────────────────
print('=== COVID Largo vs Control — Pruebas Estadísticas ===')
for metrica in ['puntuacion_PHQ9','puntuacion_GAD7','puntuacion_cognitiva','puntuacion_fatiga']:
    cl = df[df.estado=='COVID_Largo'][metrica]
    ct = df[df.estado=='Control'][metrica]
    t, p = ttest_ind(cl, ct)
    d = (cl.mean()-ct.mean()) / np.sqrt((cl.std()**2+ct.std()**2)/2)
    print(f'{metrica:28s}: CL media={cl.mean():.2f}, CT media={ct.mean():.2f}, '
          f't={t:.2f}, p={p:.3e}, d={d:.2f}')

## Cita
```bibtex
@data{DVN/QUWC1F_2024,
  author    = {de la Serna, Juan Moises},
  title     = {COVID Largo: Síntomas Neurológicos y Psiquiátricos — Prevalencia Global 2021-2024},
  year      = {2024},
  publisher = {Harvard Dataverse},
  doi       = {10.7910/DVN/QUWC1F},
  url       = {https://doi.org/10.7910/DVN/QUWC1F}
}
```
## Licencia: CC0 1.0 Universal — Dominio Público